<a href="https://colab.research.google.com/github/AbdullahRFA/Structured-Data-Extraction-Tool-Calling-Telecom-Agent/blob/main/TeleSupport_AI_Abdullah_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment Setup & Dependencies

In [1]:
# Step 0: Install required libraries
!pip install -q openai pydantic langchain langchain-openai

# Section 1: Question 1 — Structured Customer Data Extraction

## 1.1 Define the Pydantic Schema

In [2]:
import json
from typing import Literal
from pydantic import BaseModel, Field


class CustomerTicket(BaseModel):
    """Structured representation of a telecom customer support message."""

    name: str = Field(
        ...,
        description="The name of the customer. Use 'Unknown' if not explicitly stated.",
    )
    phone: str = Field(
        ...,
        description="The customer's phone number. Use 'Unknown' if not provided.",
    )
    location: str = Field(
        ...,
        description="The physical location or area mentioned by the customer. Use 'Unknown' if not provided.",
    )
    category: Literal[
        "BALANCE",
        "INTERNET",
        "RECHARGE",
        "SIM",
        "COMPLAINT",
        "OTHER",
    ] = Field(
        ...,
        description="The main category of the customer query or issue.",
    )
    sentiment: Literal["Positive", "Negative", "Neutral"] = Field(
        ...,
        description="The overall emotional tone expressed in the message.",
    )
    urgency: int = Field(
        ...,
        ge=1,
        le=5,
        description="Urgency scale from 1 (lowest) to 5 (highest/critical).",
    )
    summary: str = Field(
        ...,
        description="A concise one-sentence summary of the user message.",
    )


# Generate and print the JSON schema
ticket_schema = CustomerTicket.model_json_schema()
print(json.dumps(ticket_schema, indent=2))

{
  "description": "Structured representation of a telecom customer support message.",
  "properties": {
    "name": {
      "description": "The name of the customer. Use 'Unknown' if not explicitly stated.",
      "title": "Name",
      "type": "string"
    },
    "phone": {
      "description": "The customer's phone number. Use 'Unknown' if not provided.",
      "title": "Phone",
      "type": "string"
    },
    "location": {
      "description": "The physical location or area mentioned by the customer. Use 'Unknown' if not provided.",
      "title": "Location",
      "type": "string"
    },
    "category": {
      "description": "The main category of the customer query or issue.",
      "enum": [
        "BALANCE",
        "INTERNET",
        "RECHARGE",
        "SIM",
        "COMPLAINT",
        "OTHER"
      ],
      "title": "Category",
      "type": "string"
    },
    "sentiment": {
      "description": "The overall emotional tone expressed in the message.",
      "enum": [
 

## Question 1.2: Structured Extraction with Cloud Model

### Cloud API Key Verification & Model Discovery

In [3]:
from google.colab import userdata
from openai import OpenAI

gemini_api_key = userdata.get("GEMINI_API_KEY")
assert gemini_api_key, "GEMINI_API_KEY not found! Add it in Colab Secrets (🔑) first."

test_client = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# List available models to verify auth & pick a valid model ID for your account
models = [m.id for m in test_client.models.list().data]
print("Connected! Sample models:", models[:10])

Connected! Sample models: ['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite']


### Cloud Structured Extraction Execution (5 Test Cases)

In [5]:
import time
from google.colab import userdata
from openai import APIStatusError, OpenAI

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY in Colab Secrets.")

# Keep retries short per model so fallback triggers quickly if a model is overloaded
cloud_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    timeout=30.0,
    max_retries=1,
)

MODEL_CANDIDATES = [
    "gemini-3.8-flash",
    "gemini-3.5-flash",
    "gemini-3-flash-preview",
    "gemini-flash-latest",
]

SYSTEM_PROMPT = (
    "You are a precise telecom support data extraction assistant. "
    "Extract customer details from the message strictly according to the schema. "
    "Use 'Unknown' for missing name, phone, or location. "
    "Write a concise one-sentence summary."
)


def _call_structured_for_model(model: str, message: str) -> CustomerTicket:
    """Attempt structured parsing via parse(), falling back to json_object + Pydantic validation."""
    try:
        completion = cloud_client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": message},
            ],
            response_format=CustomerTicket,
            reasoning_effort="low",
            temperature=0,
        )
        ticket = completion.choices[0].message.parsed
        if ticket is not None:
            return ticket
        raw_content = completion.choices[0].message.content or "{}"
        return CustomerTicket.model_validate_json(raw_content)
    except APIStatusError as e:
        # Re-raise 503 / 429 / server errors so the outer loop can try the next model
        if e.status_code in (429, 500, 502, 503, 504):
            raise
        # For non-transient schema/parse errors, try json_object mode on the same model
        completion = cloud_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT + "\nRespond ONLY with valid JSON."},
                {"role": "user", "content": message},
            ],
            response_format={"type": "json_object"},
            reasoning_effort="low",
            temperature=0,
        )
        raw_content = completion.choices[0].message.content or "{}"
        return CustomerTicket.model_validate_json(raw_content)


def extract_ticket_cloud(message: str, candidates: list[str] = MODEL_CANDIDATES) -> tuple[CustomerTicket, float, str]:
    """Extract a validated CustomerTicket across candidate cloud models with fallback."""
    start_time = time.perf_counter()
    last_err = None

    for model in candidates:
        try:
            ticket = _call_structured_for_model(model, message)
            latency_sec = time.perf_counter() - start_time
            return ticket, latency_sec, model
        except Exception as e:
            last_err = e
            print(f"  [Fallback] '{model}' failed ({type(e).__name__}: {e}). Trying next model...", flush=True)

    raise RuntimeError(f"All cloud models failed. Last error: {last_err}")


test_cases = [
    (1, "My name is Sayem, number 01712345678, I live in Uttara. No network since morning, this is unacceptable!", "Name: Sayem"),
    (2, "Hi, I'm Nusrat from Chattogram. My SIM stopped working after I dropped my phone. 01812345678", "Name: Nusrat"),
    (3, "How much balance do I have left? My number is 01912345678.", "Name: Unknown"),
    (4, "I was charged twice for the same data pack. Fix this now! - Abir, 01912345678, Mirpur", "Name: Abir"),
    (5, "Thanks, the new 4G speed is amazing! I am very happy with the service.", "Name: Unknown"),
]

cloud_results = []

for case_id, msg, expected_note in test_cases:
    print(f"Processing Test #{case_id}...", flush=True)
    ticket, latency, used_model = extract_ticket_cloud(msg)
    cloud_results.append({
        "id": case_id,
        "message": msg,
        "expected": expected_note,
        "model": used_model,
        "ticket": ticket,
        "latency_sec": round(latency, 3),
    })
    print(f"=== Test #{case_id} ({expected_note}) | Model: {used_model} | Latency: {latency:.3f}s ===")
    print(ticket.model_dump_json(indent=2))
    print()

Processing Test #1...
  [Fallback] 'gemini-3.8-flash' failed (InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]). Trying next model...
=== Test #1 (Name: Sayem) | Model: gemini-3.5-flash | Latency: 3.311s ===
{
  "name": "Sayem",
  "phone": "01712345678",
  "location": "Uttara",
  "category": "INTERNET",
  "sentiment": "Negative",
  "urgency": 4,
  "summary": "Sayem from Uttara reported having no network coverage since the morning and expressed frustration."
}

Processing Test #2...
  [Fallback] 'gemini-3.8-flash' failed (InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]). Trying next model...
=== Test #2 (Name: Nusrat) | Model: gemini-3.5-flash | Latenc

## Step 1.3A: Install & Start Ollama in Colab

In [7]:
import subprocess
import time
import urllib.request

# 1. Install zstd (required by Ollama's Linux installer) and pciutils
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils > /dev/null

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Verify ollama binary exists
!ollama --version

# 4. Start Ollama server in the background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# 5. Wait until Ollama API responds
for _ in range(25):
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as resp:
            if resp.status == 200:
                print("Ollama server is up!", flush=True)
                break
    except Exception:
        time.sleep(1)

# 6. Pull gemma4:e4b (with fallback to gemma4:e2b or gemma3:1b if needed)
OLLAMA_MODEL = "gemma4:e4b"
print(f"Pulling '{OLLAMA_MODEL}'...", flush=True)
pull_cmd = subprocess.run(["ollama", "pull", OLLAMA_MODEL], capture_output=True, text=True)
if pull_cmd.returncode != 0:
    print(f"Could not pull '{OLLAMA_MODEL}': {pull_cmd.stderr.strip()}")
    for fallback_model in ["gemma4:e2b", "gemma3:1b"]:
        print(f"Trying fallback '{fallback_model}'...", flush=True)
        fb = subprocess.run(["ollama", "pull", fallback_model], capture_output=True, text=True)
        if fb.returncode == 0:
            OLLAMA_MODEL = fallback_model
            break

print("Ready with Ollama model:", OLLAMA_MODEL)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd servi

## Step 1.3B: Run the 5 Test Messages on Ollama & Compare With Cloud

In [10]:
import json
import subprocess
import time
import urllib.request
from openai import OpenAI


def ensure_ollama_running(model_name: str = "gemma4:e4b") -> str:
    """Ensure Ollama server is listening on 127.0.0.1:11434 and model is available."""

    def is_up() -> bool:
        try:
            with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
                return resp.status == 200
        except Exception:
            return False

    proc = None
    if not is_up():
        print("Ollama not listening on 127.0.0.1:11434. Starting 'ollama serve'...", flush=True)
        log_file = open("/tmp/ollama_serve.log", "w")
        proc = subprocess.Popen(
            ["ollama", "serve"],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        for _ in range(30):
            if is_up():
                print("Ollama server is now listening!", flush=True)
                break
            if proc.poll() is not None:
                log_file.flush()
                with open("/tmp/ollama_serve.log") as f:
                    print("ollama serve exited early! Logs:\n" + f.read())
                raise RuntimeError("Ollama server failed to stay running.")
            time.sleep(1)
        else:
            log_file.flush()
            with open("/tmp/ollama_serve.log") as f:
                raise RuntimeError("Timed out waiting for Ollama server. Logs:\n" + f.read())

    # Check if model is present; pull if missing
    tags_raw = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5).read().decode()
    installed_models = [m.get("name", "") for m in json.loads(tags_raw).get("models", [])]
    if model_name not in installed_models and not any(m.startswith(model_name) for m in installed_models):
        print(f"Pulling '{model_name}'...", flush=True)
        subprocess.run(["ollama", "pull", model_name], check=True)

    return model_name


OLLAMA_MODEL = ensure_ollama_running(globals().get("OLLAMA_MODEL", "gemma4:e4b"))

local_client = OpenAI(
    api_key="ollama",
    base_url="http://127.0.0.1:11434/v1/",
    timeout=120.0,
)


def extract_ticket_local(message: str, model: str = OLLAMA_MODEL) -> tuple[CustomerTicket, float]:
    """Extract a validated CustomerTicket using local Ollama via OpenAI-compatible endpoint."""
    start_time = time.perf_counter()
    schema_json = json.dumps(CustomerTicket.model_json_schema())
    local_system = (
        f"{SYSTEM_PROMPT}\n"
        f"Return ONLY valid JSON matching this schema:\n{schema_json}"
    )

    try:
        completion = local_client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": local_system},
                {"role": "user", "content": message},
            ],
            response_format=CustomerTicket,
            temperature=0,
        )
        ticket = completion.choices[0].message.parsed
        if ticket is None:
            raw_content = completion.choices[0].message.content or "{}"
            ticket = CustomerTicket.model_validate_json(raw_content)
    except Exception as e:
        print(f"  [Info] Local parse fallback ({type(e).__name__}: {e})", flush=True)
        completion = local_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": local_system},
                {"role": "user", "content": message},
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )
        raw_content = completion.choices[0].message.content or "{}"
        ticket = CustomerTicket.model_validate_json(raw_content)

    latency_sec = time.perf_counter() - start_time
    return ticket, latency_sec


local_results = []
for case_id, msg, expected_note in test_cases:
    print(f"Processing Local Test #{case_id}...", flush=True)
    ticket, latency = extract_ticket_local(msg)
    local_results.append({
        "id": case_id,
        "message": msg,
        "expected": expected_note,
        "model": OLLAMA_MODEL,
        "ticket": ticket,
        "latency_sec": round(latency, 3),
    })
    print(f"=== [Ollama: {OLLAMA_MODEL}] Test #{case_id} ({expected_note}) | Latency: {latency:.3f}s ===")
    print(ticket.model_dump_json(indent=2))
    print()

print(f"{'Test':<5} | {'Cloud (s)':<10} | {'Local (s)':<10} | {'Cloud Cat/Sent':<22} | {'Local Cat/Sent'}")
print("-" * 92)
for c_item, l_item in zip(cloud_results, local_results):
    c_meta = f"{c_item['ticket'].category}/{c_item['ticket'].sentiment}"
    l_meta = f"{l_item['ticket'].category}/{l_item['ticket'].sentiment}"
    print(
        f"#{c_item['id']:<4} | {c_item['latency_sec']:<10} | {l_item['latency_sec']:<10} | "
        f"{c_meta:<22} | {l_meta}"
    )

Ollama not listening on 127.0.0.1:11434. Starting 'ollama serve'...
Ollama server is now listening!
Processing Local Test #1...
=== [Ollama: gemma4:e4b] Test #1 (Name: Sayem) | Latency: 129.287s ===
{
  "name": "Sayem",
  "phone": "01712345678",
  "location": "Uttara",
  "category": "COMPLAINT",
  "sentiment": "Negative",
  "urgency": 5,
  "summary": "Sayem is reporting a complete lack of network service in Uttara and finds the situation unacceptable."
}

Processing Local Test #2...
=== [Ollama: gemma4:e4b] Test #2 (Name: Nusrat) | Latency: 6.744s ===
{
  "name": "Nusrat",
  "phone": "01812345678",
  "location": "Chattogram",
  "category": "SIM",
  "sentiment": "Negative",
  "urgency": 4,
  "summary": "Nusrat's SIM card stopped working after dropping the phone."
}

Processing Local Test #3...
=== [Ollama: gemma4:e4b] Test #3 (Name: Unknown) | Latency: 4.955s ===
{
  "name": "Unknown",
  "phone": "01912345678",
  "location": "Unknown",
  "category": "BALANCE",
  "sentiment": "Neutral",


## LLM Structured Extraction Benchmark: `gemini-3.5-flash` vs. `gemma4:e4b`

### Executive Summary
Both **cloud (`gemini-3.5-flash`)** and **local (`gemma4:e4b` via Ollama)** models achieved **100% schema compliance** and accurate entity extraction across all 5 customer ticket test cases. While `gemini-3.5-flash` delivered faster, more predictable latency, `gemma4:e4b` matched its extraction accuracy once warmed up—offering a viable zero-quota local alternative.

---

### Performance & Reliability Comparison

| Metric | `gemini-3.5-flash` (Cloud) | `gemma4:e4b` (Local Ollama) |
| :--- | :--- | :--- |
| **Schema Compliance** | **100%** (5/5 valid `CustomerTicket` instances) | **100%** (5/5 valid `CustomerTicket` instances) |
| **Cold-Start Latency** | N/A | **129.287s** (Test #1 model/VRAM load) |
| **Warm Latency Range** | **1.747s – 5.061s** | **4.955s – 7.383s** (Tests #2–#5) |
| **Entity Extraction** | **100%** accurate (Names, Phones, Locations, `"Unknown"` fallbacks) | **100%** accurate (Names, Phones, Locations, `"Unknown"` fallbacks) |
| **Rate Limits / Quotas** | Subject to API limits | **None** (fully local execution) |

---

### Edge-Case Classification Differences
Both models extracted exact entity values (e.g., *Sayem, Nusrat, Abir*, and `"Unknown"` for missing fields) and generated concise one-sentence summaries, diverging only on subjective classification labels:

* **Test #1 (No network + *"unacceptable!"*):**
  * **Cloud:** `INTERNET` | Urgency: `4` *(prioritized technical root cause)*
  * **Local:** `COMPLAINT` | Urgency: `5` *(prioritized explicit frustrated tone)*
* **Test #2 (SIM stopped working after dropping phone):**
  * **Cloud:** Sentiment: `Neutral` | Urgency: `3`
  * **Local:** Sentiment: `Negative` | Urgency: `4` *(better reflects a broken SIM experience)*
* **Test #4 (Charged twice for data pack):**
  * **Cloud:** `COMPLAINT` *(treated as a billing dispute)*
  * **Local:** `BALANCE` *(associated double-charge with account balance)*

---

### Key Takeaways
1. **Production Readiness:** Both models reliably enforce strict `Literal` types, integer bounds (`1..5`), and missing-value defaults.
2. **Latency vs. Cost Trade-off:** Use `gemini-3.5-flash` for real-time responsiveness; use `gemma4:e4b` for rate-limit-free batch processing (with an initial warm-up call to absorb the ~129s load time).
3. **Prompt Opportunity:** Defining explicit tie-breaker rules in the schema (e.g., prioritizing technical/billing categories over generic `COMPLAINT`) will align classification behavior across both models.


# Section 2 — Question 2: Tool-Calling AI Assistant (LangChain)

## 2.1 Define Tools

In [11]:
import uuid
from langchain_core.tools import tool


@tool
def get_balance(phone: str) -> str:
    """Return the current account balance and validity date for a customer's phone number."""
    # Mock telecom database lookup
    mock_accounts = {
        "01712345678": {"balance_bdt": 145.50, "validity": "2026-12-15"},
        "01812345678": {"balance_bdt": 320.00, "validity": "2026-12-28"},
        "01912345678": {"balance_bdt": 68.75, "validity": "2026-12-05"},
    }
    record = mock_accounts.get(phone.strip(), {"balance_bdt": 50.00, "validity": "2026-12-10"})
    return (
        f"Phone: {phone.strip()} | Balance: BDT {record['balance_bdt']:.2f} | "
        f"Valid until: {record['validity']}"
    )


@tool
def get_data_packages(validity_days: int) -> str:
    """Return available mobile data packages filtered by validity duration in days."""
    packages = [
        {"name": "Smart Starter 2GB", "data": "2 GB", "validity_days": 3, "price_bdt": 49},
        {"name": "Weekly Streamer 12GB", "data": "12 GB", "validity_days": 7, "price_bdt": 149},
        {"name": "Monthly Power 30GB", "data": "30 GB", "validity_days": 30, "price_bdt": 449},
        {"name": "Monthly Ultra 60GB + 200 Mins", "data": "60 GB + 200 mins", "validity_days": 30, "price_bdt": 699},
    ]
    matched = [p for p in packages if p["validity_days"] == validity_days]
    if not matched:
        available_days = sorted({p["validity_days"] for p in packages})
        return (
            f"No exact packages found for {validity_days}-day validity. "
            f"Available validity options: {available_days} days."
        )
    lines = [
        f"- {p['name']}: {p['data']} | Validity: {p['validity_days']} days | Price: BDT {p['price_bdt']}"
        for p in matched
    ]
    return f"Packages for {validity_days} days:\n" + "\n".join(lines)


@tool
def create_support_ticket(phone: str, issue: str, priority: int) -> str:
    """Create a telecom support ticket for a customer issue and return a unique Ticket ID."""
    clamped_priority = max(1, min(5, int(priority)))
    ticket_id = f"TKT-{uuid.uuid4().hex[:8].upper()}"
    return (
        f"Support ticket created successfully. "
        f"Ticket ID: {ticket_id} | Phone: {phone.strip()} | "
        f"Priority: {clamped_priority}/5 | Status: Open | Issue: {issue.strip()}"
    )


@tool
def check_network_status(area: str) -> str:
    """Check current network coverage, outages, or maintenance status for a specified area."""
    area_clean = area.strip()
    area_key = area_clean.lower()
    network_reports = {
        "uttara": "Advisory: Ongoing tower maintenance in Uttara Sector 7-11; intermittent 4G/5G speeds expected until 18:00.",
        "gulshan": "Normal: 4G and 5G network operating at full capacity with no reported outages.",
        "mirpur": "Minor Degradation: High evening traffic congestion reported in Mirpur-10; engineers optimizing load.",
        "chattogram": "Normal: Core network healthy; localized fiber work completed earlier today.",
    }
    status_msg = network_reports.get(
        area_key,
        f"Normal: No major outages or scheduled maintenance reported in {area_clean}.",
    )
    return f"Area: {area_clean} | {status_msg}"


TOOLS = [
    get_balance,
    get_data_packages,
    create_support_ticket,
    check_network_status,
]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
print("Registered tools:", list(TOOLS_BY_NAME.keys()))

Registered tools: ['get_balance', 'get_data_packages', 'create_support_ticket', 'check_network_status']


## 2.2 Create the Agent

### LangChain Google GenAI Dependency

In [16]:
!pip install -q langchain-google-genai

### Agent Initialization, Validation Suite & Interactive Loop

In [18]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI

TELECOM_SYSTEM_PROMPT = """You are TeleSupport AI, a courteous, efficient, and helpful telecom customer support engineer.
Guidelines:
1. For general greetings or casual questions that do not require account, package, ticket, or network lookups, respond directly in a polite, conversational manner without calling tools.
2. When a user asks for account balance/validity, use `get_balance`.
3. When a user asks about data plans/packages for a validity period, use `get_data_packages`.
4. When a user reports a service issue, outage, billing error, or requests support help with a phone number/issue, use `create_support_ticket` (choose an appropriate priority from 1 to 5 based on severity).
5. When a user asks about network coverage, outages, or maintenance in a specific area, use `check_network_status`.
6. If a user asks multiple things in one message (such as checking packages AND reporting an internet outage), invoke all relevant tools and combine the results clearly in your final response.
7. Never invent balances, ticket IDs, or network statuses—always rely on tool outputs."""

agent_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0,
)

llm_with_tools = agent_llm.bind_tools(TOOLS)


def run_agent_turn(
    user_query: str,
    chat_history: list[BaseMessage] | None = None,
    max_steps: int = 5,
    verbose: bool = True,
) -> tuple[str, list[str], list[BaseMessage]]:
    """
    Execute one user turn through the LangChain tool-calling loop.
    Returns (final_response_text, list_of_invoked_tool_names, updated_messages_for_turn).
    """
    history = list(chat_history) if chat_history else []
    messages: list[BaseMessage] = [
        SystemMessage(content=TELECOM_SYSTEM_PROMPT),
        *history,
        HumanMessage(content=user_query),
    ]
    turn_messages: list[BaseMessage] = [HumanMessage(content=user_query)]
    invoked_tools: list[str] = []

    for _ in range(max_steps):
        ai_msg: AIMessage = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        turn_messages.append(ai_msg)

        if not ai_msg.tool_calls:
            final_text = ai_msg.content if isinstance(ai_msg.content, str) else str(ai_msg.content)
            return final_text, invoked_tools, turn_messages

        for tool_call in ai_msg.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})
            tool_call_id = tool_call["id"]
            invoked_tools.append(tool_name)

            if verbose:
                print(f"  [Tool Call] {tool_name}({tool_args})", flush=True)

            selected_tool = TOOLS_BY_NAME.get(tool_name)
            if selected_tool is None:
                tool_output = f"Error: Tool '{tool_name}' is not registered."
            else:
                tool_output = selected_tool.invoke(tool_args)

            if verbose:
                print(f"  [Tool Result] {tool_output}", flush=True)

            tool_msg = ToolMessage(content=str(tool_output), tool_call_id=tool_call_id)
            messages.append(tool_msg)
            turn_messages.append(tool_msg)

    return "I apologize, I reached the maximum number of tool steps while resolving your request.", invoked_tools, turn_messages


validation_prompts = [
    (1, "Hey, how are you?", "Direct conversation (No tool call)"),
    (2, "What's the balance for 01712345678?", "Tool: get_balance"),
    (3, "Show me 30-day packages, and my internet has been down for 2 days, my number is 01812345678", "Multi-Tool: get_data_packages, create_support_ticket"),
    (4, "Is there any network problem in Uttara?", "Tool: check_network_status"),
]

validation_logs = []
session_history: list[BaseMessage] = []

for test_num, prompt_text, expected_path in validation_prompts:
    print("=" * 80)
    print(f"Validation #{test_num} | Expected: {expected_path}")
    print(f"User: {prompt_text}")
    reply, tools_used, turn_msgs = run_agent_turn(prompt_text, chat_history=session_history, verbose=True)
    session_history.extend(turn_msgs)
    validation_logs.append({
        "test_num": test_num,
        "prompt": prompt_text,
        "expected_path": expected_path,
        "tools_invoked": tools_used,
        "response": reply,
    })
    print(f"Assistant: {reply}\n")


def run_interactive_agent() -> None:
    """Interactive loop accepting user queries via input() until user types 'exit'."""
    print("=== TeleSupport AI Interactive Assistant (type 'exit' to quit) ===")
    loop_history: list[BaseMessage] = []
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() == "exit":
            print("Assistant: Thank you for contacting TeleSupport AI. Goodbye!")
            break
        if not user_input:
            continue
        reply, _, turn_msgs = run_agent_turn(user_input, chat_history=loop_history, verbose=True)
        loop_history.extend(turn_msgs)
        print(f"Assistant: {reply}")

Validation #1 | Expected: Direct conversation (No tool call)
User: Hey, how are you?
Assistant: [{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I assist you with your telecom services today?", 'extras': {'signature': 'EskDCsYDAWkUfROfJRvl2HuJUpx3D02ipwDmiEJlb86iPUPzUFGs+WavrhhX21O4m1SHLiIrOw+MxpumZSaMKGYDg6lhdN5iJ/RNByIdtLL2xV5dN/7TArbB35vW7SoCJOH/7tDNr/ETXVkCBFKM+VH29YiNrIZxSieSknGLsMqCLSKsilzXPNvjYrYRbd0YvL0LjuSv5WGLp5a76m0eqDOhGWfILDAMWasmW8p/Fvwcy3mzd7JaMjO5BbnoulwmQdU81DIN0+WIaVxCgVOwQ/vJjVWXkriOrItGZAChKYEMFobfVhZ1rMTomMXPtc0Cqv4ENhxwGbG4SsjtzvHPmBegUKSnCO/pjeSqE0K5M2WJfouT+E83ifRtzC6SFywrhqauXy/AU990N1j3iCQxc88GS7T8vehSIIiPIUKELITfBEQc+PZbnGMiz5MiWXouO34qMbqOeKa2cGrDwwt2LY3P0A7j8b4c/vTw2fAFazJ/SLXCY+zi+H9ZG/CG1A3N8qB9JoXjCSh+WCuFyYpYN38rIg3sL/Hm8ar46n1CSF0vHc3jCwKqQR9Uvi/YPos0qYajS8AjS51G9FmJK28VDPtb0fiykavnw81H+A=='}}]

Validation #2 | Expected: Tool: get_balance
User: What's the balance for 01712345678?
  [Tool Call] get_balance({'phone': '01712

### Response Text Normalization Helper

In [19]:
def extract_text_content(content) -> str:
    """Normalize LangChain AIMessage content (str or list of content blocks) to plain text."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            elif isinstance(block, str):
                parts.append(block)
        return "\n".join(p for p in parts if p).strip()
    return str(content)

# Section 3: Final Demonstration Logs

In [24]:
import ast


def clean_log_response(resp) -> str:
    """Extract plain text whether resp is a string, block list, or stringified block list."""
    if isinstance(resp, str) and resp.strip().startswith("["):
        try:
            resp = ast.literal_eval(resp)
        except Exception:
            return resp.strip()
    if isinstance(resp, list):
        parts = [
            b.get("text", "") if isinstance(b, dict) and b.get("type") == "text" else str(b)
            for b in resp
        ]
        return "\n".join(p.strip() for p in parts if p and p.strip()).strip()
    return str(resp).strip()


print("=" * 90)
print("SECTION 3: FINAL DEMONSTRATION LOGS — TELESUPPORT AI")
print("=" * 90)

print("\n--- [1.2 & 1.3] Structured Extraction Summary (Cloud vs. Local Ollama) ---")
for c_item, l_item in zip(cloud_results, local_results):
    print(f"\nTest #{c_item['id']}: \"{c_item['message']}\" ({c_item['expected']})")
    print(f"  Cloud ({c_item['model']}, {c_item['latency_sec']}s): {c_item['ticket'].model_dump()}")
    print(f"  Local ({l_item['model']}, {l_item['latency_sec']}s): {l_item['ticket'].model_dump()}")

print("\n--- [2.3] Tool-Calling Agent Validation Summary ---")
for v in validation_logs:
    print(f"\nValidation #{v['test_num']} | Expected: {v['expected_path']} | Invoked: {v['tools_invoked']}")
    print(f"  User: {v['prompt']}")
    print(f"  Assistant: {clean_log_response(v['response'])}")

if globals().get("custom_interactive_logs"):
    print("\n--- [2.3] Custom Interactive Session Log ---")
    for turn in custom_interactive_logs:
        print(f"\nTurn #{turn['turn']} | Invoked: {turn['tools_invoked']}")
        print(f"  User: {turn['prompt']}")
        print(f"  Assistant: {clean_log_response(turn['response'])}")

SECTION 3: FINAL DEMONSTRATION LOGS — TELESUPPORT AI

--- [1.2 & 1.3] Structured Extraction Summary (Cloud vs. Local Ollama) ---

Test #1: "My name is Sayem, number 01712345678, I live in Uttara. No network since morning, this is unacceptable!" (Name: Sayem)
  Cloud (gemini-3.5-flash, 3.311s): {'name': 'Sayem', 'phone': '01712345678', 'location': 'Uttara', 'category': 'INTERNET', 'sentiment': 'Negative', 'urgency': 4, 'summary': 'Sayem from Uttara reported having no network coverage since the morning and expressed frustration.'}
  Local (gemma4:e4b, 129.287s): {'name': 'Sayem', 'phone': '01712345678', 'location': 'Uttara', 'category': 'COMPLAINT', 'sentiment': 'Negative', 'urgency': 5, 'summary': 'Sayem is reporting a complete lack of network service in Uttara and finds the situation unacceptable.'}

Test #2: "Hi, I'm Nusrat from Chattogram. My SIM stopped working after I dropped my phone. 01812345678" (Name: Nusrat)
  Cloud (gemini-3.5-flash, 5.061s): {'name': 'Nusrat', 'phone': '0181

### Custom Interactive Agent Session

In [23]:
from langchain_core.messages import BaseMessage

custom_interactive_logs: list[dict] = []


def extract_text_content(content) -> str:
    """Normalize LangChain AIMessage content (str or content blocks) into clean plain text."""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            elif isinstance(block, str):
                parts.append(block)
        return "\n".join(p.strip() for p in parts if p and p.strip()).strip()
    return str(content).strip()


def run_interactive_agent(
    record_logs: bool = True,
    show_tool_traces: bool = True,
) -> list[dict]:
    """
    Interactive telecom support loop using input() until user types 'exit'.
    Supports '/reset' to clear chat history and '/summary' to view session turns.
    """
    print("=" * 75)
    print("TeleSupport AI — Interactive Mode")
    print("Commands: type 'exit' to quit | '/reset' to clear memory | '/summary' for turn count")
    print("=" * 75)

    loop_history: list[BaseMessage] = []
    turn_index = 0

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nAssistant: Session interrupted. Goodbye!")
            break

        if not user_input:
            continue

        cmd = user_input.lower()
        if cmd in ("exit", "quit"):
            print("Assistant: Thank you for using TeleSupport AI. Have a great day!")
            break

        if cmd == "/reset":
            loop_history.clear()
            print("[System] Conversation history cleared.")
            continue

        if cmd == "/summary":
            print(f"[System] Turns completed this session: {turn_index} | History messages: {len(loop_history)}")
            continue

        turn_index += 1
        raw_reply, tools_used, turn_msgs = run_agent_turn(
            user_input,
            chat_history=loop_history,
            verbose=show_tool_traces,
        )
        clean_reply = extract_text_content(raw_reply)
        loop_history.extend(turn_msgs)

        print(f"Assistant: {clean_reply}")

        if record_logs:
            custom_interactive_logs.append({
                "turn": turn_index,
                "prompt": user_input,
                "tools_invoked": tools_used,
                "response": clean_reply,
            })

    return custom_interactive_logs


# Start the interactive session
run_interactive_agent(record_logs=True, show_tool_traces=True)

TeleSupport AI — Interactive Mode
Commands: type 'exit' to quit | '/reset' to clear memory | '/summary' for turn count

You: checking balance
Assistant: [{'type': 'text', 'text': 'To help you check your account balance, could you please provide your phone number?', 'extras': {'signature': 'Eu8JCuwJAWkUfRM/1jXmOwk18NhzFLPmPzKCuKljNnYU0TYzyYrGI81+AlxvInzUW5VbHb4AKaEHLaL/v7IRVL8hJTNBzah81DFSZTpVdnVHaETdgvnQueEZJPKPx3B0hvrrxY8Ul+zbOuAe6DqAmEgYrYM2Q6slUDwmyd4X6+D0RtSjUCaBKelNFa0mnIwry5100IhCIQy3+7q6RIzD66WFrTVnkKbBOGAoK0tza14sBjFrW4Lx0FD+4OdJyIrxPC2SPPa3W9C/4/+BqtFQmWC5DQQLfEEPAqs1XQajDkl7TScnlzi1wze3LtQdxTNg0qb8pwIxgtxmWPdjZ2tYWQsFjmJ9QW7d+oIA8IiWuGzD64R+sKtxaatZrpfpaIHuT2QStsdd43FqOtH511SKvDUZQWNdgnMfG3B5QWF+FOqb2nV1/XXH/hcKTiI4MQWe0etPsDbSnvy6OJ5gSJMdO7XG9oTJ0Nf7u9+sjIXaCqZxbMao8rmyMDoexdbMqUtLM5vcggXJtoAZmD4lelDio7UhMVJfJ/TECneuXUhbTiMhXjUIY6jvS9oosh7/Fr91RmHPdPEO5BLObnP8gwZEK8mJKyo9x0ESRbpbg84yBd6Xy199nkIMRv2I3yuA9pFIlbSagfZbPVXB8SPAQ/6CybOTFIUuYBFHCRyaY4AJ0EKZxkRgrjEmkMZtsTlnzfCpObvDX

[{'turn': 1,
  'prompt': 'checking balance',
  'tools_invoked': [],
  'response': "[{'type': 'text', 'text': 'To help you check your account balance, could you please provide your phone number?', 'extras': {'signature': 'Eu8JCuwJAWkUfRM/1jXmOwk18NhzFLPmPzKCuKljNnYU0TYzyYrGI81+AlxvInzUW5VbHb4AKaEHLaL/v7IRVL8hJTNBzah81DFSZTpVdnVHaETdgvnQueEZJPKPx3B0hvrrxY8Ul+zbOuAe6DqAmEgYrYM2Q6slUDwmyd4X6+D0RtSjUCaBKelNFa0mnIwry5100IhCIQy3+7q6RIzD66WFrTVnkKbBOGAoK0tza14sBjFrW4Lx0FD+4OdJyIrxPC2SPPa3W9C/4/+BqtFQmWC5DQQLfEEPAqs1XQajDkl7TScnlzi1wze3LtQdxTNg0qb8pwIxgtxmWPdjZ2tYWQsFjmJ9QW7d+oIA8IiWuGzD64R+sKtxaatZrpfpaIHuT2QStsdd43FqOtH511SKvDUZQWNdgnMfG3B5QWF+FOqb2nV1/XXH/hcKTiI4MQWe0etPsDbSnvy6OJ5gSJMdO7XG9oTJ0Nf7u9+sjIXaCqZxbMao8rmyMDoexdbMqUtLM5vcggXJtoAZmD4lelDio7UhMVJfJ/TECneuXUhbTiMhXjUIY6jvS9oosh7/Fr91RmHPdPEO5BLObnP8gwZEK8mJKyo9x0ESRbpbg84yBd6Xy199nkIMRv2I3yuA9pFIlbSagfZbPVXB8SPAQ/6CybOTFIUuYBFHCRyaY4AJ0EKZxkRgrjEmkMZtsTlnzfCpObvDXDZQLec8St+SwteZwDpHG/GEYbOflzZWUxEG3h4UqyLmk3z37K1Tkhx5NBUztr4i6inzun4